In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# Load original dataset
df = pd.read_csv("crop_yield.csv")

target = "Yield_tons_per_hectare"
groups = df["Region"]

baseline_features = df.drop(columns=[target])

# Feature groups
feature_groups = {
    "Weather Removed": ["Weather_Condition"],
    "Soil Removed": ["Soil_Type"],
    "Crop Removed": ["Crop"],
    "Fertilizer & Irrigation Removed": ["Fertilizer_Used", "Irrigation_Used"],
    "Temperature Removed": ["Temperature_Celsius"],
    "Rainfall Removed": ["Rainfall_mm"],
    "Days_to_Harvest Removed": ["Days_to_Harvest"]
}

def evaluate_model(X, y, groups):
    gkf = GroupKFold(n_splits=4)
    r2_scores = []

    categorical_cols = [col for col in ["Region","Soil_Type","Crop","Weather_Condition"] if col in X.columns]
    numerical_cols = [col for col in ["Rainfall_mm","Temperature_Celsius","Days_to_Harvest"] if col in X.columns]
    binary_cols = [col for col in ["Fertilizer_Used","Irrigation_Used"] if col in X.columns]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("bin", "passthrough", binary_cols)
        ]
    )

    model = LinearRegression()

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        r2_scores.append(r2_score(y_test, y_pred))

    return np.mean(r2_scores)

# Baseline
baseline_r2 = evaluate_model(baseline_features, df[target], groups)

results = [("Baseline (All Features)", baseline_r2)]

# Ablation tests
for name, cols_to_remove in feature_groups.items():
    X_modified = baseline_features.drop(columns=cols_to_remove)
    r2_value = evaluate_model(X_modified, df[target], groups)
    results.append((name, r2_value))

results_df = pd.DataFrame(results, columns=["Feature Configuration", "Mean R2"])
print(results_df)

             Feature Configuration   Mean R2
0          Baseline (All Features)  0.912975
1                  Weather Removed  0.912976
2                     Soil Removed  0.912975
3                     Crop Removed  0.912975
4  Fertilizer & Irrigation Removed  0.591969
5              Temperature Removed  0.905785
6                 Rainfall Removed  0.327264
7          Days_to_Harvest Removed  0.912975
